### Allscripts Sunrise (SCM) Measurement Hydration

Numeric SCM observations promoted into `measurement`.

#### Notes on the current approach
- `dbo_cv3observationcur` contains both true measurements and a large amount of narrative/status content.
- The prior implementation filtered on numeric-looking `ValueText` first, which still scanned too broadly and let non-measurement content dominate runtime.
- This version filters to rows with an explicit OMOP `Measurement` mapping before numeric parsing.
- Numeric extraction is now defensive: it accepts simple comparator-prefixed values like `<5` and `>100`, while still excluding rows that do not yield a numeric value.
- A dedupe step is applied before the silver merge because `cv3observationdocumentcur` can duplicate the same observation across document links.

In [0]:
%sql
-- Reset Gold
TRUNCATE TABLE _exponent.omop_scm.measurement;


In [0]:
%sql
-- Reset Silver
DELETE FROM _exponent.omop_silver.measurement
WHERE source_system = 'allscripts_scm';


In [0]:
%sql
-- Reset Mapping
DELETE FROM _exponent.omop_mapping.source_to_measurement
WHERE source_system = 'allscripts_scm';


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW silver_measurement_stage AS
WITH mapped_measurement_rows AS (
  SELECT
    obs.GUID AS observation_guid,
    stp.person_id,
    meas_concept.omop_concept_id AS measurement_concept_id,
    CAST(COALESCE(obsdoc.RecordedDtm, doc.AuthoredDtm, obs.ETL_LOAD_TS) AS DATE) AS measurement_date,
    COALESCE(obsdoc.RecordedDtm, doc.AuthoredDtm, obs.ETL_LOAD_TS) AS measurement_datetime,
    NULL AS measurement_time,
    32817 AS measurement_type_concept_id,
    CASE
      WHEN TRIM(obs.ValueText) RLIKE '^<' THEN 4171755
      WHEN TRIM(obs.ValueText) RLIKE '^>' THEN 4172704
      WHEN TRIM(obs.ValueText) RLIKE '^<=' THEN 4171754
      WHEN TRIM(obs.ValueText) RLIKE '^>=' THEN 4171756
      ELSE NULL
    END AS operator_concept_id,
    TRY_CAST(REGEXP_EXTRACT(TRIM(obs.ValueText), '(-?[0-9]+(?:\\.[0-9]+)?)', 1) AS DOUBLE) AS value_as_number,
    NULL AS value_as_concept_id,
    COALESCE(unit_concept.concept_id, 0) AS unit_concept_id,
    NULL AS range_low,
    NULL AS range_high,
    stpr.provider_id AS provider_id,
    stvo.visit_occurrence_id AS visit_occurrence_id,
    NULL AS visit_detail_id,
    CONCAT_WS(CHR(31), 'allscripts_scm', 'dbo_cv3observationcur', 'GUID', CAST(obs.GUID AS STRING)) AS measurement_source_value,
    0 AS measurement_source_concept_id,
    obs.UnitOfMeasure AS unit_source_value,
    0 AS unit_source_concept_id,
    obs.ValueText AS value_source_value,
    NULL AS measurement_event_id,
    NULL AS meas_event_field_concept_id,
    'allscripts_scm' AS source_system,
    CURRENT_TIMESTAMP() AS last_mod_tsp
  FROM _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationcur obs
  INNER JOIN _exponent.omop_mapping.domain_source_to_concept meas_concept
    ON meas_concept.source_id = CAST(obs.ObsItemGUID AS STRING)
   AND meas_concept.domain_id = 'Measurement'
   AND meas_concept.source_system = 'allscripts_scm'
   AND meas_concept.active_flag = TRUE
  INNER JOIN _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationdocumentcur obsdoc
    ON obs.GUID = obsdoc.ObservationGUID
  INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientdocumentcur doc
    ON obsdoc.OwnerGUID = doc.GUID
  INNER JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(doc.ClientGUID AS STRING))
   AND stp.active_flag = TRUE
  LEFT JOIN _exponent.omop_mapping.source_to_provider stpr
    ON stpr.provider_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'dbo_cv3careprovider', 'GUID', CAST(doc.AuthoredProviderGUID AS STRING))
   AND stpr.active_flag = TRUE
  LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
    ON stvo.visit_occurrence_source_value = CONCAT('allscripts_scm', ' | ', CAST(doc.ClientVisitGUID AS STRING))
   AND stvo.active_flag = TRUE
  LEFT JOIN _exponent.omop.concept unit_concept
    ON unit_concept.vocabulary_id = 'UCUM'
   AND unit_concept.concept_code = CASE
     WHEN TRIM(obs.UnitOfMeasure) = 'percent' THEN '%'
     WHEN TRIM(obs.UnitOfMeasure) = '%' THEN '%'
     WHEN TRIM(obs.UnitOfMeasure) = 'mm Hg' THEN 'mm[Hg]'
     WHEN TRIM(obs.UnitOfMeasure) = 'mmHg' THEN 'mm[Hg]'
     WHEN TRIM(obs.UnitOfMeasure) = 'oC' THEN 'Cel'
     WHEN TRIM(obs.UnitOfMeasure) = 'Degrees C' THEN 'Cel'
     WHEN TRIM(obs.UnitOfMeasure) = 'degrees C' THEN 'Cel'
     WHEN TRIM(obs.UnitOfMeasure) = 'C' THEN 'Cel'
     WHEN TRIM(obs.UnitOfMeasure) = '°C' THEN 'Cel'
     WHEN TRIM(obs.UnitOfMeasure) = 'Degrees F' THEN '[degF]'
     WHEN TRIM(obs.UnitOfMeasure) = 'degrees F' THEN '[degF]'
     WHEN TRIM(obs.UnitOfMeasure) = 'F' THEN '[degF]'
     WHEN TRIM(obs.UnitOfMeasure) = '°F' THEN '[degF]'
     WHEN TRIM(obs.UnitOfMeasure) = 'ml' THEN 'mL'
     WHEN TRIM(obs.UnitOfMeasure) = 'mL' THEN 'mL'
     WHEN TRIM(obs.UnitOfMeasure) = 'milliLiter(s)' THEN 'mL'
     WHEN TRIM(obs.UnitOfMeasure) = 'Milliliter(s)' THEN 'mL'
     WHEN TRIM(obs.UnitOfMeasure) = 'seconds' THEN 's'
     WHEN TRIM(obs.UnitOfMeasure) = 'Second(s)' THEN 's'
     WHEN TRIM(obs.UnitOfMeasure) = 'sec' THEN 's'
     WHEN TRIM(obs.UnitOfMeasure) = 'Minute(s)' THEN 'min'
     WHEN TRIM(obs.UnitOfMeasure) = 'Hour(s)' THEN 'h'
     WHEN TRIM(obs.UnitOfMeasure) = 'Day(s)' THEN 'd'
     WHEN TRIM(obs.UnitOfMeasure) = 'Day' THEN 'd'
     WHEN TRIM(obs.UnitOfMeasure) = 'Week(s)' THEN 'wk'
     WHEN TRIM(obs.UnitOfMeasure) = 'lb' THEN '[lb_us]'
     WHEN TRIM(obs.UnitOfMeasure) = 'Pound(s)' THEN '[lb_us]'
     WHEN TRIM(obs.UnitOfMeasure) = 'Ounce(s)' THEN '[oz_av]'
     WHEN TRIM(obs.UnitOfMeasure) = 'Gm' THEN 'g'
     WHEN TRIM(obs.UnitOfMeasure) = 'Gram(s)' THEN 'g'
     WHEN TRIM(obs.UnitOfMeasure) = 'Centimeter(s)' THEN 'cm'
     WHEN TRIM(obs.UnitOfMeasure) = 'Cm' THEN 'cm'
     WHEN TRIM(obs.UnitOfMeasure) = 'Inch(s)' THEN '[in_i]'
     WHEN TRIM(obs.UnitOfMeasure) = 'Feet' THEN '[ft_i]'
     WHEN TRIM(obs.UnitOfMeasure) = 'Meter Squared' THEN 'm2'
     WHEN TRIM(obs.UnitOfMeasure) = 'kG/m2' THEN 'kg/m2'
     WHEN TRIM(obs.UnitOfMeasure) = 'bpm' THEN '/min'
     WHEN TRIM(obs.UnitOfMeasure) = 'RPM' THEN '/min'
     WHEN TRIM(obs.UnitOfMeasure) = 'Breaths/Min' THEN '/min'
     WHEN TRIM(obs.UnitOfMeasure) = 'Breaths/min' THEN '/min'
     WHEN TRIM(obs.UnitOfMeasure) = 'mL/Hr' THEN 'mL/h'
     WHEN TRIM(obs.UnitOfMeasure) = 'mL/Min' THEN 'mL/min'
     WHEN TRIM(obs.UnitOfMeasure) = 'L/Min' THEN 'L/min'
     WHEN TRIM(obs.UnitOfMeasure) = 'mL/beat' THEN 'mL/{beat}'
     WHEN TRIM(obs.UnitOfMeasure) = 'mL/beat/m2' THEN 'mL/{beat}/m2'
     WHEN TRIM(obs.UnitOfMeasure) = 'cm H2O' THEN 'cm[H2O]'
     WHEN TRIM(obs.UnitOfMeasure) = 'Watts' THEN 'W'
     WHEN TRIM(obs.UnitOfMeasure) = 'Unit(s)' THEN '[arb''U]'
     WHEN TRIM(obs.UnitOfMeasure) = 'ratio' THEN '{ratio}'
     ELSE TRIM(obs.UnitOfMeasure)
   END
  WHERE obs.GUID IS NOT NULL
    AND obs.StatusType = 1
    AND obs.ValueText IS NOT NULL
    AND COALESCE(obsdoc.RecordedDtm, doc.AuthoredDtm, obs.ETL_LOAD_TS) >= TIMESTAMP('1950-01-01')
), deduped AS (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY measurement_source_value
           ORDER BY measurement_datetime DESC, visit_occurrence_id DESC, provider_id DESC
         ) AS rn
  FROM mapped_measurement_rows
  WHERE value_as_number IS NOT NULL
)
SELECT
  person_id,
  measurement_concept_id,
  measurement_date,
  measurement_datetime,
  measurement_time,
  measurement_type_concept_id,
  operator_concept_id,
  value_as_number,
  value_as_concept_id,
  unit_concept_id,
  range_low,
  range_high,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  measurement_source_value,
  measurement_source_concept_id,
  unit_source_value,
  unit_source_concept_id,
  value_source_value,
  measurement_event_id,
  meas_event_field_concept_id,
  source_system,
  last_mod_tsp
FROM deduped
WHERE rn = 1;

In [0]:
%sql
MERGE INTO _exponent.omop_silver.measurement AS t
USING silver_measurement_stage AS s
ON t.measurement_source_value = s.measurement_source_value

WHEN MATCHED AND (
     NOT (t.person_id <=> s.person_id)
  OR NOT (t.measurement_concept_id <=> s.measurement_concept_id)
  OR NOT (t.measurement_date <=> s.measurement_date)
  OR NOT (t.measurement_datetime <=> s.measurement_datetime)
  OR NOT (t.measurement_type_concept_id <=> s.measurement_type_concept_id)
  OR NOT (t.operator_concept_id <=> s.operator_concept_id)
  OR NOT (t.value_as_number <=> s.value_as_number)
  OR NOT (t.provider_id <=> s.provider_id)
  OR NOT (t.visit_occurrence_id <=> s.visit_occurrence_id)
  OR NOT (t.unit_source_value <=> s.unit_source_value)
  OR NOT (t.value_source_value <=> s.value_source_value)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.person_id = s.person_id,
  t.measurement_concept_id = s.measurement_concept_id,
  t.measurement_date = s.measurement_date,
  t.measurement_datetime = s.measurement_datetime,
  t.measurement_time = s.measurement_time,
  t.measurement_type_concept_id = s.measurement_type_concept_id,
  t.operator_concept_id = s.operator_concept_id,
  t.value_as_number = s.value_as_number,
  t.value_as_concept_id = s.value_as_concept_id,
  t.unit_concept_id = s.unit_concept_id,
  t.range_low = s.range_low,
  t.range_high = s.range_high,
  t.provider_id = s.provider_id,
  t.visit_occurrence_id = s.visit_occurrence_id,
  t.visit_detail_id = s.visit_detail_id,
  t.measurement_source_concept_id = s.measurement_source_concept_id,
  t.unit_source_value = s.unit_source_value,
  t.unit_source_concept_id = s.unit_source_concept_id,
  t.value_source_value = s.value_source_value,
  t.measurement_event_id = s.measurement_event_id,
  t.meas_event_field_concept_id = s.meas_event_field_concept_id,
  t.source_system = s.source_system,
  t.last_mod_tsp = s.last_mod_tsp

WHEN NOT MATCHED THEN INSERT (
  person_id,
  measurement_concept_id,
  measurement_date,
  measurement_datetime,
  measurement_time,
  measurement_type_concept_id,
  operator_concept_id,
  value_as_number,
  value_as_concept_id,
  unit_concept_id,
  range_low,
  range_high,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  measurement_source_value,
  measurement_source_concept_id,
  unit_source_value,
  unit_source_concept_id,
  value_source_value,
  measurement_event_id,
  meas_event_field_concept_id,
  source_system,
  last_mod_tsp
)
VALUES (
  s.person_id,
  s.measurement_concept_id,
  s.measurement_date,
  s.measurement_datetime,
  s.measurement_time,
  s.measurement_type_concept_id,
  s.operator_concept_id,
  s.value_as_number,
  s.value_as_concept_id,
  s.unit_concept_id,
  s.range_low,
  s.range_high,
  s.provider_id,
  s.visit_occurrence_id,
  s.visit_detail_id,
  s.measurement_source_value,
  s.measurement_source_concept_id,
  s.unit_source_value,
  s.unit_source_concept_id,
  s.value_source_value,
  s.measurement_event_id,
  s.meas_event_field_concept_id,
  s.source_system,
  s.last_mod_tsp
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_measurement (
  source_system,
  measurement_source_value,
  active_flag,
  created_tsp,
  last_mod_tsp,
  merge_id,
  merge_reason
)
SELECT
  s.source_system,
  s.measurement_source_value,
  TRUE,
  CURRENT_TIMESTAMP(),
  COALESCE(s.last_mod_tsp, CURRENT_TIMESTAMP()),
  NULL,
  NULL
FROM _exponent.omop_silver.measurement s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_measurement x
  ON s.measurement_source_value = x.measurement_source_value
WHERE s.source_system = 'allscripts_scm';

In [0]:
%sql
MERGE INTO _exponent.omop_scm.measurement AS gold
USING (
  SELECT
    stm.measurement_id,
    s.person_id,
    s.measurement_concept_id,
    s.measurement_date,
    s.measurement_datetime,
    s.measurement_time,
    s.measurement_type_concept_id,
    s.operator_concept_id,
    s.value_as_number,
    s.value_as_concept_id,
    s.unit_concept_id,
    s.range_low,
    s.range_high,
    s.provider_id,
    s.visit_occurrence_id,
    s.visit_detail_id,
    s.measurement_source_value,
    s.measurement_source_concept_id,
    s.unit_source_value,
    s.value_source_value
  FROM _exponent.omop_silver.measurement s
  JOIN _exponent.omop_scm.person p
    ON p.person_id = s.person_id
  JOIN _exponent.omop_mapping.source_to_measurement stm
    ON stm.measurement_source_value = s.measurement_source_value
   AND stm.source_system = 'allscripts_scm'
   AND stm.active_flag = TRUE
  WHERE s.source_system = 'allscripts_scm'
) AS src
ON gold.measurement_id = src.measurement_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id = src.person_id,
  gold.measurement_concept_id = src.measurement_concept_id,
  gold.measurement_date = src.measurement_date,
  gold.measurement_datetime = src.measurement_datetime,
  gold.measurement_time = src.measurement_time,
  gold.measurement_type_concept_id = src.measurement_type_concept_id,
  gold.operator_concept_id = src.operator_concept_id,
  gold.value_as_number = src.value_as_number,
  gold.value_as_concept_id = src.value_as_concept_id,
  gold.unit_concept_id = src.unit_concept_id,
  gold.range_low = src.range_low,
  gold.range_high = src.range_high,
  gold.provider_id = src.provider_id,
  gold.visit_occurrence_id = src.visit_occurrence_id,
  gold.visit_detail_id = src.visit_detail_id,
  gold.measurement_source_value = src.measurement_source_value,
  gold.measurement_source_concept_id = src.measurement_source_concept_id,
  gold.unit_source_value = src.unit_source_value,
  gold.value_source_value = src.value_source_value

WHEN NOT MATCHED THEN INSERT (
  measurement_id,
  person_id,
  measurement_concept_id,
  measurement_date,
  measurement_datetime,
  measurement_time,
  measurement_type_concept_id,
  operator_concept_id,
  value_as_number,
  value_as_concept_id,
  unit_concept_id,
  range_low,
  range_high,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  measurement_source_value,
  measurement_source_concept_id,
  unit_source_value,
  value_source_value
)
VALUES (
  src.measurement_id,
  src.person_id,
  src.measurement_concept_id,
  src.measurement_date,
  src.measurement_datetime,
  src.measurement_time,
  src.measurement_type_concept_id,
  src.operator_concept_id,
  src.value_as_number,
  src.value_as_concept_id,
  src.unit_concept_id,
  src.range_low,
  src.range_high,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.measurement_source_value,
  src.measurement_source_concept_id,
  src.unit_source_value,
  src.value_source_value
);

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN unit_source_value IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_unit_source,
  SUM(CASE WHEN unit_concept_id <> 0 THEN 1 ELSE 0 END) AS rows_with_unit_concept
FROM _exponent.omop_silver.measurement
WHERE source_system = 'allscripts_scm';